# Credit Suisse PCA Monitor — USD SOFR Non-Overlapping Forwards

**Strategy:** Single PCA on non-overlapping forward rates. Residual dashboard, NxN hedge ratio matrix, portfolio factor decomposition.

**Reference:** Credit Suisse — "PCA Unleashed" (Pelata, Giannopoulos, Haworth — Oct 2012)

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.pca_rv_engine import PCARVConfig, PCARVResult, rolling_pca, adf_test
from BT.signals.pca_hedge_ratios import (
    hedge_ratio_matrix, portfolio_factor_exposure,
    portfolio_variance_decomposition, suggested_hedge,
)
from BT.signals.rv_backtest import RVBacktestConfig, run_rv_backtest

In [ ]:
config = {
    # === DATA ===
    "forward_tenors": [
        ("Spot", "1Y"),   # spot 1Y
        ("1Y", "1Y"),     # 1y1y
        ("2Y", "1Y"),     # 2y1y
        ("3Y", "1Y"),     # 3y1y
        ("4Y", "1Y"),     # 4y1y
        ("5Y", "1Y"),     # 5y1y
        ("6Y", "1Y"),     # 6y1y
        ("7Y", "1Y"),     # 7y1y
        ("8Y", "1Y"),     # 8y1y
        ("9Y", "1Y"),     # 9y1y
        ("10Y", "2Y"),    # 10y2y
        ("12Y", "3Y"),    # 12y3y
        ("15Y", "5Y"),    # 15y5y
        ("20Y", "5Y"),    # 20y5y
        ("25Y", "5Y"),    # 25y5y
        ("30Y", "10Y"),   # 30y10y
    ],
    "data_start": "2021-01-01",
    "data_end": None,
    # === PCA ===
    "pca_window_days": 260,             # 1Y rolling (Credit Suisse default)
    "pca_window_options": [130, 260, 520],
    "pca_input": "levels",
    "n_components": 3,
    "use_covariance": True,
    # === RESIDUAL ANALYSIS ===
    "residual_history_days": 260,
    "residual_snapshot_lookbacks": [1, 5, 22],  # days ago to overlay
    # === HEDGE RATIOS ===
    "hedge_neutralize_factors": [1, 2],  # PC1+PC2 = level-and-slope-neutral
    # === PORTFOLIO TOOL ===
    "portfolio_positions": {},  # user fills in DV01 exposures
    # === BACKTEST (lightweight) ===
    "backtest_enabled": True,
    "backtest_fly_candidates": [
        ("1Y1Y", "2Y1Y", "3Y1Y"),
        ("2Y1Y", "5Y1Y", "9Y1Y"),
        ("5Y1Y", "10Y2Y", "20Y5Y"),
        ("10Y2Y", "20Y5Y", "30Y10Y"),
    ],
    "backtest_entry_residual_threshold_bp": 3.0,
    "backtest_exit_residual_crosses_zero": True,
    "backtest_exit_max_days": 30,
    "backtest_exit_stop_bp": 6.0,
}

In [ ]:
curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
irswaps_tb = IRSwapsTB(curve_mdp, show_tqdm=True)
tb = TimeseriesBuilder(irswaps_tb=irswaps_tb)

end_date = config["data_end"] or datetime.date.today()
start_date = datetime.datetime.strptime(config["data_start"], "%Y-%m-%d").date()

queries = []
tenor_labels = []
for fwd, swap_tenor in config["forward_tenors"]:
    if fwd == "Spot":
        q = IRSwapQuery(curve="USD-SOFR-1D", tenor=swap_tenor, value=IRSwapValue.RATE)
        label = f"spot_{swap_tenor.lower()}"
    else:
        q = IRSwapQuery(curve="USD-SOFR-1D", tenor=f"{fwd}{swap_tenor}", value=IRSwapValue.RATE)
        label = f"{fwd.lower()}{swap_tenor.lower()}"
    queries.append(q)
    tenor_labels.append(label)

fwd_rates_df = tb.get_timeseries(start=start_date, end=end_date, queries=queries, n_jobs=8)

# Rename to short labels
col_map = dict(zip(fwd_rates_df.columns, tenor_labels))
fwd_rates_df = fwd_rates_df.rename(columns=col_map)
print(f"Loaded: {fwd_rates_df.shape}")
fwd_rates_df.tail(3)

In [ ]:
pca_config = PCARVConfig(
    pca_window_days=config["pca_window_days"],
    pca_input=config["pca_input"],
    n_components=config["n_components"],
    use_correlation=not config["use_covariance"],
    zscore_lookback_days=config["pca_window_days"],
)

pca_result = rolling_pca(fwd_rates_df.dropna(axis=0, how="any"), pca_config)
print(f"PCA complete. Latest VE: {pca_result.variance_explained.dropna().iloc[-1].values}")

## Residual Dashboard

In [ ]:
latest = fwd_rates_df.index[-1]
lookbacks = config["residual_snapshot_lookbacks"]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(tenor_labels))
width = 0.6 / (len(lookbacks) + 1)

# Current residuals
current_res = pca_result.residuals.loc[latest].values * 10000  # in bp
ax.bar(x, current_res, width=width, label="Today", color="steelblue", zorder=5)

# Lookback overlays
colors = ["orange", "green", "red"]
for k, lb in enumerate(lookbacks):
    lb_date = fwd_rates_df.index[max(0, len(fwd_rates_df) - 1 - lb)]
    if lb_date in pca_result.residuals.index:
        lb_res = pca_result.residuals.loc[lb_date].values * 10000
        ax.bar(x + (k + 1) * width, lb_res, width=width,
               label=f"{lb}d ago ({lb_date.strftime('%m/%d')})",
               color=colors[k % len(colors)], alpha=0.6)

ax.set_xticks(x + width)
ax.set_xticklabels(tenor_labels, rotation=45, fontsize=9)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_ylabel("Residual (bp)")
ax.set_title(f"PCA Residual Snapshot — USD SOFR Non-Overlapping Forwards ({latest.strftime('%Y-%m-%d')})\n"
             f"+ = Cheap, − = Rich")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
n_tenors = len(tenor_labels)
n_cols = 4
n_rows = (n_tenors + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows), sharex=True)
axes = axes.flatten()

history_days = config["residual_history_days"]
for i, col in enumerate(pca_result.residuals.columns):
    ax = axes[i]
    res = pca_result.residuals[col].iloc[-history_days:] * 10000
    ax.plot(res.index, res.values, linewidth=0.8, color="steelblue")
    mean = res.mean()
    std = res.std()
    ax.axhline(y=mean, color="black", linewidth=0.5, linestyle="--")
    ax.axhline(y=mean + std, color="gray", linewidth=0.5, linestyle=":")
    ax.axhline(y=mean - std, color="gray", linewidth=0.5, linestyle=":")
    ax.axhline(y=mean + 2 * std, color="red", linewidth=0.5, linestyle=":")
    ax.axhline(y=mean - 2 * std, color="red", linewidth=0.5, linestyle=":")
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.suptitle("Residual Time Series (bp) with ±1σ and ±2σ Bands")
plt.tight_layout()
plt.show()

In [ ]:
latest_dt = sorted(pca_result.loadings.keys())[-1]
ldg = pca_result.loadings[latest_dt]
ve = pca_result.variance_explained.loc[latest_dt]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
x = np.arange(len(tenor_labels))

# Variance explained
axes[0].bar(range(len(ve)), ve.values * 100)
axes[0].set_xticks(range(len(ve)))
axes[0].set_xticklabels(ve.index)
axes[0].set_title("Variance Explained (%)")
axes[0].set_ylabel("%")

# PC1/PC2/PC3 loading shapes
for k in range(3):
    axes[k + 1].bar(x, ldg[:, k])
    axes[k + 1].set_xticks(x)
    axes[k + 1].set_xticklabels(tenor_labels, rotation=45, fontsize=7)
    axes[k + 1].set_title(f"PC{k+1} Loadings ({ve.iloc[k]*100:.1f}%)")
    axes[k + 1].axhline(y=0, color="black", linewidth=0.5)

plt.suptitle(f"PCA Decomposition — {latest_dt.strftime('%Y-%m-%d')}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot PC1/PC2/PC3 scores over time
scores_df = pd.DataFrame(pca_result.scores).T
scores_df.columns = [f"PC{i+1}" for i in range(scores_df.shape[1])]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
pc_labels = ["PC1 (Level)", "PC2 (Slope)", "PC3 (Curvature)"]
colors = ["blue", "green", "orange"]
for k in range(3):
    axes[k].plot(scores_df.index, scores_df.iloc[:, k], color=colors[k], linewidth=0.8)
    axes[k].set_ylabel(pc_labels[k])
    axes[k].grid(True, alpha=0.3)
plt.suptitle("PC Score Time Series")
plt.tight_layout()
plt.show()

## Hedge Ratio Matrix & Portfolio Decomposition

In [ ]:
ldg_df = pd.DataFrame(ldg, index=tenor_labels, columns=[f"PC{i+1}" for i in range(ldg.shape[1])])
evals = pca_result.eigenvalues[latest_dt]
evals_series = pd.Series(evals, index=[f"PC{i+1}" for i in range(len(evals))])

hrm = hedge_ratio_matrix(ldg_df, evals_series, neutralize_factors=config["hedge_neutralize_factors"])

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(hrm, annot=True, fmt=".2f", cmap="RdYlGn_r", center=1.0, ax=ax,
            linewidths=0.5, vmin=-2, vmax=4)
factor_str = "+".join([f"PC{f}" for f in config["hedge_neutralize_factors"]])
ax.set_title(f"Hedge Ratio Matrix ({factor_str}-Neutral) — {latest_dt.strftime('%Y-%m-%d')}\n"
             f"Row x, Col y: units of y needed to hedge 1 unit of x")
plt.tight_layout()
plt.show()

In [ ]:
if config["portfolio_positions"]:
    positions = pd.Series(config["portfolio_positions"])
    exposure = portfolio_factor_exposure(positions, ldg_df)
    decomp = portfolio_variance_decomposition(positions, ldg_df, evals_series)

    print("Factor Exposures:")
    display(exposure)
    print(f"\nVariance Decomposition:")
    for k, v in decomp.items():
        print(f"  {k}: {v:.2f}")

    # Pie chart
    pc_vars = {k: v for k, v in decomp.items() if k != "total"}
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(pc_vars.values(), labels=pc_vars.keys(), autopct="%1.1f%%")
    ax.set_title("Portfolio Variance by Factor")
    plt.show()
else:
    print("No portfolio positions specified — skipping factor decomposition.")
    print("Set config['portfolio_positions'] = {'1y1y': -10, '5y1y': 50, ...} to use.")

In [ ]:
actual = fwd_rates_df.loc[latest].values * 100  # percent
recon = pca_result.reconstructed.loc[latest].values * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = range(len(tenor_labels))

axes[0].plot(x, actual, "o-", label="Actual", color="blue")
axes[0].plot(x, recon, "s--", label="PCA Reconstructed", color="red")
axes[0].set_xticks(x)
axes[0].set_xticklabels(tenor_labels, rotation=45, fontsize=8)
axes[0].set_title("Actual vs PCA-Reconstructed Forward Rates")
axes[0].set_ylabel("Rate (%)")
axes[0].legend()

# Residual bar chart
res_bp = (actual - recon) * 100  # bp
axes[1].bar(x, res_bp, color=["green" if r > 0 else "red" for r in res_bp])
axes[1].set_xticks(x)
axes[1].set_xticklabels(tenor_labels, rotation=45, fontsize=8)
axes[1].set_title("Residual = Actual - Reconstructed (bp)")
axes[1].axhline(y=0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

## Divergence Detection & Backtest

In [ ]:
def detect_divergences(residuals, threshold_bp=3.0):
    """Flag adjacent tenors with opposite-sign residuals exceeding threshold."""
    latest = residuals.iloc[-1] * 10000  # bp
    flags = []
    cols = list(residuals.columns)
    for i in range(len(cols) - 1):
        r1, r2 = latest[cols[i]], latest[cols[i + 1]]
        if np.sign(r1) != np.sign(r2) and abs(r1) > threshold_bp and abs(r2) > threshold_bp:
            flags.append({
                "tenor_1": cols[i], "residual_1_bp": r1,
                "tenor_2": cols[i + 1], "residual_2_bp": r2,
                "spread_bp": abs(r1) + abs(r2),
                "signal": "butterfly_candidate",
            })
    return pd.DataFrame(flags).sort_values("spread_bp", ascending=False) if flags else pd.DataFrame()

divergences = detect_divergences(pca_result.residuals)
if len(divergences) > 0:
    print("Adjacent-tenor divergences detected:")
    display(divergences)
else:
    print("No adjacent-tenor divergences above threshold.")

In [ ]:
def structural_break_warning(residuals, lookback_days=60, trend_threshold=0.7):
    """Flag tenors where residuals are trending (not mean-reverting).

    Uses correlation of residual with time as a simple trend detector.
    High |correlation| -> trending -> not a mean-reverting trade opportunity.
    """
    recent = residuals.iloc[-lookback_days:]
    warnings = []
    t = np.arange(len(recent))
    for col in recent.columns:
        r = recent[col].dropna().values
        if len(r) < lookback_days // 2:
            continue
        corr = np.corrcoef(t[:len(r)], r)[0, 1]
        if abs(corr) > trend_threshold:
            warnings.append({
                "tenor": col,
                "time_corr": corr,
                "direction": "trending_cheap" if corr > 0 else "trending_rich",
                "warning": "Possible structural shift - NOT a mean-reverting opportunity",
            })
    return pd.DataFrame(warnings)

breaks = structural_break_warning(pca_result.residuals)
if len(breaks) > 0:
    print("Structural break warnings (residual trending, not mean-reverting):")
    display(breaks)
else:
    print("No structural break warnings detected.")

In [ ]:
if config["backtest_enabled"]:
    from BT.signals.pca_rv_engine import pca_fly_weights as pfw

    bt_residuals, bt_zscores, bt_rsq, bt_weights, bt_rates = {}, {}, {}, {}, {}
    fly_cats = {}

    for fly_tuple in config["backtest_fly_candidates"]:
        fly_id = "/".join(fly_tuple)
        cols = [t.lower() for t in fly_tuple]
        # Find matching columns in fwd_rates_df
        matching = [c for c in fwd_rates_df.columns if any(t in c for t in cols)]
        if len(matching) == 3:
            df3 = fwd_rates_df[matching].dropna()
            pca_cfg = PCARVConfig(
                pca_window_days=config["pca_window_days"],
                pca_input=config["pca_input"], n_components=3,
            )
            fly_result = rolling_pca(df3, pca_cfg)
            bt_residuals[fly_id] = fly_result.residuals.iloc[:, 1]  # belly residual
            bt_zscores[fly_id] = fly_result.zscores.iloc[:, 1]
            bt_rsq[fly_id] = fly_result.variance_explained.sum(axis=1)
            bt_weights[fly_id] = pfw(df3, pca_cfg)
            bt_rates[fly_id] = df3.rename(columns=dict(zip(matching, ["left", "belly", "right"])))
            fly_cats[fly_id] = "non_overlapping_fwd"

    if bt_residuals:
        regime = pd.Series("green", index=fwd_rates_df.index)
        bt_config = RVBacktestConfig(
            mtm_mode="approximate",
            entry_min_residual_bp=config["backtest_entry_residual_threshold_bp"],
            entry_min_zscore=1.0,
            entry_min_rsq=0.0,
            exit_mean_reversion=config["backtest_exit_residual_crosses_zero"],
            exit_max_holding_days=config["backtest_exit_max_days"],
        )
        bt_result = run_rv_backtest(
            bt_residuals, bt_zscores, bt_rsq, bt_weights, bt_rates,
            regime, fly_cats, config=bt_config,
        )
        print(f"Backtest: {bt_result.metrics['n_trades']} trades, "
              f"Hit rate: {bt_result.metrics['hit_rate']:.1%}, "
              f"Sharpe: {bt_result.metrics['sharpe']:.2f}")

        bt_result.cumulative_pnl.plot(figsize=(12, 4), title="Cumulative P&L (bp)")
        plt.grid(True, alpha=0.3)
        plt.show()